# RNN for Classifying Names

In this notebook we are building and training a basic character-level RNN to classify
words. A character-level RNN reads words as a series of characters -
outputting a prediction and "hidden state" at each step, feeding its
previous hidden state into each next time step. We take the final prediction
to be the output, i.e. which class the word belongs to.

### Preparing the Data

Download the data in folder `data/names` from GitHub.

Included in the ``data/names`` directory are 18 text files named as
``[Language].txt``. Each file contains a bunch of names, one name per
line, mostly romanized (but we still need to convert from Unicode to
ASCII).

We first get all the filenames:

In [9]:
import glob
filenames = glob.glob('data/names/*.txt')

print(filenames)

['data/names/Czech.txt', 'data/names/German.txt', 'data/names/Arabic.txt', 'data/names/Japanese.txt', 'data/names/Chinese.txt', 'data/names/Vietnamese.txt', 'data/names/Russian.txt', 'data/names/French.txt', 'data/names/Irish.txt', 'data/names/English.txt', 'data/names/Spanish.txt', 'data/names/Greek.txt', 'data/names/Italian.txt', 'data/names/Portuguese.txt', 'data/names/Scottish.txt', 'data/names/Dutch.txt', 'data/names/Korean.txt', 'data/names/Polish.txt']


And save each language as a category:

In [10]:
import os
all_categories = []


for filename in filenames:
    language = os.path.splitext(os.path.basename(filename))[0]
    all_categories.append(language)

print(all_categories)

['Czech', 'German', 'Arabic', 'Japanese', 'Chinese', 'Vietnamese', 'Russian', 'French', 'Irish', 'English', 'Spanish', 'Greek', 'Italian', 'Portuguese', 'Scottish', 'Dutch', 'Korean', 'Polish']


Next we load the data and put every name in a list together and its category (=label) in a second list:

In [11]:
X = []
y = []


for index, filename in enumerate(filenames):
    lines = open(filename, encoding='utf-8').read().strip().split('\n')
    category = all_categories[index]
    for line in lines:
        X.append(line)
        y.append(category)

n_categories = len(all_categories)
n_categories, len(X)

(18, 20074)

Let's check which characters are included in the names:

In [12]:
all_characters = set([c for name in X for c in name])
print(all_characters)
print(len(all_characters), "characters")

{'a', 'H', 'Á', 'j', '1', 'd', 'á', 'é', 'D', 'í', 'ł', 'Ż', 'i', 's', 'w', 'm', 'p', 'E', 'G', 'ó', 'Ś', 'e', 'ń', 'Q', 'õ', 'J', 'v', 'L', ',', 'u', 'è', 'y', 'A', 'ü', 'F', 'x', 'h', 'V', 'g', ' ', 'ö', 'ñ', 'k', 'l', 'C', 'P', 'n', 'ż', '-', 'Z', 'U', 'ù', 'ì', 'b', 'R', 'ä', 'W', ':', 'r', 'ç', 'T', 'ą', 'O', 'ã', 'S', 'à', 'ß', 'f', '\xa0', 'ò', 'o', 'ú', 't', 'Y', 'z', 'K', 'É', 'c', 'ê', '/', 'B', 'N', "'", 'X', 'M', 'q', 'I'}
87 characters


We see that the files contain many special characters that make our problem more difficult. To reduce the character count, we only allow ASCII symbols:

In [13]:
import string

# these is the vocabulary we will use
all_letters = string.ascii_letters
n_letters = len(all_letters)

print(f"Vocab is of size {n_letters} and contains:", all_letters)

Vocab is of size 52 and contains: abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ


In [14]:
import unicodedata

# this method converts anything into ascii
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in all_letters
    )

print(unicodeToAscii('Ślusàrski'))
print(unicodeToAscii('Frühling'))

Slusarski
Fruhling


In [15]:
# convert all letters to ascii
X = [unicodeToAscii(x) for x in X]

# print again all characters
all_characters = set([c for name in X for c in name])
print(all_characters)
print(len(all_characters), "characters")

{'a', 'H', 'j', 'b', 'J', 'R', 'd', 'v', 'L', 'f', 'W', 'u', 'o', 'y', 't', 'A', 'Y', 'z', 'F', 'D', 'x', 'h', 'V', 'g', 'K', 'c', 'B', 'i', 'k', 'r', 's', 'T', 'l', 'N', 'w', 'C', 'O', 'P', 'M', 'S', 'q', 'X', 'n', 'm', 'p', 'E', 'G', 'e', 'Q', 'Z', 'U', 'I'}
52 characters


We can see that we successfully reduced the number of characters and can now divide the data into train and test data:

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

print("Train data points:", len(X_train))

Train data points: 16059


Turning Names into Tensors
--------------------------

Now that we have all the names organized, we need to turn them into
Tensors to make any use of them.

To represent a single letter, we use a "one-hot vector" of size
``<1 x n_letters>``. A one-hot vector is filled with 0s except for a 1
at index of the current letter, e.g. ``"b" = <0 1 0 0 0 ...>``.

To make a word we join a bunch of those into a 2D matrix
``<line_length x 1 x n_letters>``.

That extra 1 dimension is because PyTorch assumes everything is in
batches - we're just using a batch size of 1 here.




In [17]:
import torch

def letterToTensor(letter):
    tensor = torch.zeros(1, n_letters)
    index = all_letters.find(letter)
    tensor[0][index] = 1
    return tensor

Know lets check how the encoding of one letter looks like:

In [18]:
print(letterToTensor('J'))

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])


We also need to convert the label into a number, which is just the index of the category:

In [19]:
def categoryToTensor(category):
    index = all_categories.index(category)
    return torch.tensor([index], dtype=torch.long)

categoryToTensor("Korean")

tensor([16])

Creating the RNN
====================

This RNN module has two linear layers. One calculates the next hidden state, the other one the output.

In [20]:
import torch.nn as nn

class RNN(nn.Module):
    def __init__(self, input_size, output_size):
        super(RNN, self).__init__()

        self.hidden_size = 128 # number of hidden layer size

        self.input2hidden = nn.Linear(input_size + self.hidden_size, self.hidden_size)
        self.input2output = nn.Linear(input_size + self.hidden_size, output_size)

    def forward(self, x, hidden):
        combined = torch.cat((x, hidden), 1) # input and hidden state are combined
        hidden = self.input2hidden(combined) # calculate next hidden state
        output = self.input2output(combined) # calculate output state
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, self.hidden_size)

To run a step of this network we need to pass an input (in our case, the
tensor for the current letter) and a previous hidden state (which we
initialize as zeros at first). We get back the output and a next hidden state (which we keep for the next
step).




In [21]:
rnn = RNN(n_letters, n_categories)

x = letterToTensor('A')
hidden = rnn.initHidden()

output, next_hidden = rnn(x, hidden)
print(torch.softmax(output, 1))

tensor([[0.0529, 0.0558, 0.0573, 0.0545, 0.0559, 0.0584, 0.0588, 0.0588, 0.0568,
         0.0598, 0.0549, 0.0555, 0.0511, 0.0530, 0.0538, 0.0537, 0.0512, 0.0577]],
       grad_fn=<SoftmaxBackward0>)


As you can see the output is a ``<1 x n_categories>`` Tensor, where
every item is the likelihood of that category (higher is more likely).




Task 1: Training the Network
--------------------

Finish the following training function to train the RNN on the training data set.

In [22]:
import math


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(rnn.parameters(), lr=0.005)

for epoch in range(1, 10):
    print("Training epoch:", epoch)
    # iterate through all names in X_train
    # for every name:
        # init the hidden layer of the rnn
        # insert the name character by character into the rnn and compute the final output
        # note: you need to carry on the hidden state in every time step
        # define the loss on the last output of the rnn and the category (=label)
        # backpropagate the loss and take an optimizer step

Training epoch: 1
Training epoch: 2
Training epoch: 3
Training epoch: 4
Training epoch: 5
Training epoch: 6
Training epoch: 7
Training epoch: 8
Training epoch: 9


### Task 2: Evaluating the Results

Evaluate the accuarcy of the RNN on the test data.

### Task 3: Running on User Input

Write a function that takes an abritrary name as input and outputs the top 3 categories of the RNN for the input.
